# Validation of the LLM-assisted institutional-domain classification

Reproducible check of the GPT-5-mini domain classifier used in the AI-in-diabetes temporal
institutional network analysis (Reviewer comment: *"Add a short validation note: agreement on a
small hand-labeled sample …"*).

**Design.** 100 affiliation strings were drawn at random (`seed = 42`) from the 67,973 unique
affiliations and independently re-labeled by a domain expert who was **blinded to the model
output**. For each affiliation the expert chose one of the 12 institutional domains, or left it
unassigned to denote no specific domain ("Other"). This notebook recomputes agreement between
expert and model labels.

**Input.** `AI_Diabetes_LLM_Classification_Validation_n100_labeled.xlsx`, sheet `Validation`:
column *Your domain* = expert label, *AI domain* = model label, *AI confidence* = model confidence.

Run top-to-bottom to reproduce every number reported in Supplementary Methods S2.

In [1]:
import pandas as pd, numpy as np
from sklearn.metrics import cohen_kappa_score

XLSX = "AI_Diabetes_LLM_Classification_Validation_n100_labeled.xlsx"
df = pd.read_excel(XLSX, sheet_name="Validation")
df = df.rename(columns={df.columns[2]: "expert", df.columns[3]: "model", df.columns[4]: "confidence"})
df["expert"] = df["expert"].fillna("").astype(str).str.strip()
df["model"]  = df["model"].astype(str).str.strip()
print("rows:", len(df),
      "| expert-labeled:", int((df.expert != "").sum()),
      "| left blank:", int((df.expert == "").sum()))

rows: 100 | expert-labeled: 73 | left blank: 27


## Handling unassigned (blank) entries

The expert assigned a specific domain to **73** affiliations and left **27** unassigned. Of the 27
blanks, **22** are affiliations the model labeled `Other` (the expert left "Other"/no-fit cases
blank rather than picking "Other" from the dropdown); the remaining **5** are cases the model gave
a specific domain (genuine skips / uncertain). We report three transparent rules:

* **A (primary)** — blank scored as `Other`; all 100 affiliations included.
* **B** — expert explicit labels only; blanks excluded (n = 73).
* **C** — blank + model-`Other` scored as agreement; the 5 non-Other blanks excluded (n = 95).

In [2]:
def score(expert, model, label):
    expert = np.asarray(expert); model = np.asarray(model)
    agree = int((expert == model).sum()); n = len(expert)
    kappa = cohen_kappa_score(expert, model)
    print(f"{label:52s} n={n:3d}   agreement = {agree}/{n} = {100*agree/n:4.1f}%   Cohen's kappa = {kappa:.3f}")
    return dict(rule=label, n=n, agree=agree, pct=round(100*agree/n, 1), kappa=round(kappa, 3))

A = df.assign(e=df.expert.replace("", "Other"))
B = df[df.expert != ""]
C = df[(df.expert != "") | (df.model == "Other")].assign(e=lambda d: d.expert.replace("", "Other"))

results = [
    score(A.e,      A.model, "A. blank = 'Other' (primary)"),
    score(B.expert, B.model, "B. explicit labels only (blanks excluded)"),
    score(C.e,      C.model, "C. blank+model-Other = agreement; 5 skips excluded"),
]

A. blank = 'Other' (primary)                         n=100   agreement = 84/100 = 84.0%   Cohen's kappa = 0.811
B. explicit labels only (blanks excluded)            n= 73   agreement = 62/73 = 84.9%   Cohen's kappa = 0.830
C. blank+model-Other = agreement; 5 skips excluded   n= 95   agreement = 84/95 = 88.4%   Cohen's kappa = 0.865


## Agreement by model confidence

The classifier emits a confidence score; strings below 0.70 were flagged to a review queue in the
main pipeline. Agreement should be higher above that threshold if the score is informative.

In [3]:
a = df.assign(e=df.expert.replace("", "Other"))
for sub, lab in [(a[a.confidence >= 0.70], "confidence >= 0.70"),
                 (a[a.confidence <  0.70], "confidence <  0.70")]:
    agree = int((sub.e == sub.model).sum()); n = len(sub)
    print(f"{lab:20s} n={n:3d}   agreement = {agree}/{n} = {100*agree/n:.1f}%")

confidence >= 0.70   n= 72   agreement = 65/72 = 90.3%
confidence <  0.70   n= 28   agreement = 19/28 = 67.9%


## Per-domain agreement (primary rule): of the affiliations the model assigned to each label, how many the expert confirmed

In [4]:
a = df.assign(e=df.expert.replace("", "Other"))
rows = []
for lab, g in a.groupby("model"):
    rows.append((lab, int((g.e == g.model).sum()), len(g)))
per = pd.DataFrame(rows, columns=["model_label", "confirmed", "n"]).sort_values("n", ascending=False)
per["pct"] = (100 * per.confirmed / per.n).round(0)
print(per.to_string(index=False))

                       model_label  confirmed  n   pct
                             Other         22 32  69.0
      Technical / AI & informatics         15 16  94.0
                 Clinical medicine         10 13  77.0
          Endocrinology & diabetes          8  8 100.0
            Ophthalmology & vision          7  7 100.0
      Public health & epidemiology          4  5  80.0
Industry / digital health / device          4  4 100.0
   Complications & cardiometabolic          4  4 100.0
      Pediatrics & maternal health          3  4  75.0
        Behavioral & psychological          3  3 100.0
      Nursing & diabetes education          2  2 100.0
              Exercise & lifestyle          1  1 100.0
                         Nutrition          1  1 100.0


## Disagreements (primary rule)

In [5]:
a = df.assign(e=df.expert.replace("", "Other"))
dis = a[a.e != a.model][[df.columns[0], "Affiliation (review this)", "expert", "model", "confidence"]]
dis = dis.rename(columns={df.columns[0]: "#", "Affiliation (review this)": "affiliation",
                          "expert": "expert_or_blank"})
dis["affiliation"] = dis["affiliation"].str.slice(0, 80)
print("model said 'Other' but expert gave a domain:", int(((a.model=='Other') & (a.expert!='')).sum()))
print("model gave a domain but expert left blank   :", int(((a.model!='Other') & (a.expert=='')).sum()))
dis

model said 'Other' but expert gave a domain: 10
model gave a domain but expert left blank   : 5


,#,affiliation,expert_or_blank,model,confidence
1,2,"Data Management Department, King Abdullah Inte...",Technical / AI & informatics,Other,0.00
4,5,"a Division of Maternal-Fetal Medicine , Gineme...",Clinical medicine,Pediatrics & maternal health,0.95
7,8,"School of Health and Social Care, Edinburgh Na...",Public health & epidemiology,Other,0.00
9,10,"Department of General Surgery, Peking Universi...",,Clinical medicine,0.90
11,12,Paris-Descartes University.,Ophthalmology & vision,Other,0.00
18,19,"Centre for Health Economics, Monash University...",,Public health & epidemiology,0.90
45,46,Department of Laboratory Medicine and Patholog...,,Clinical medicine,0.90
56,57,"Department of Pediatrics, Division of Pediatri...",Endocrinology & diabetes,Other,0.00
58,59,"College of Medicine, Helse Nord TB Initiative,...",Clinical medicine,Other,0.00
69,70,Modeling and Intelligent Control Engineering L...,,Technical / AI & informatics,0.98


## Summary

Under the primary rule the model's label matched the expert's in **84/100 (84%), Cohen's κ = 0.81**
— high agreement across the 13-category scheme. Agreement was **90%** for assignments the model made
with confidence ≥ 0.70 versus **68%** below that threshold, supporting the confidence-based review
queue. Specific domains were near-perfect (technical/AI 15/16, endocrinology 8/8, ophthalmology 7/7);
disagreements concentrated in the residual **"Other"** category, where the model was occasionally
more conservative than the expert. Restricting to explicitly labeled affiliations gives 85% (κ = 0.83);
excluding the 5 unadjudicated skips gives 88% (κ = 0.87).

## Provenance — how the 100-row sample was drawn (documentation)

```python
import pandas as pd
src = pd.read_csv("data/llm/results/affiliation_classifications.csv",
                  usecols=["affiliation_text", "label", "confidence", "status"])   # 67,973 rows
sample = src.sample(n=100, random_state=42).reset_index(drop=True)                 # fixed seed
```
